# Pulser-Gym: Interactive Demo

This demonstration illustrates the integration of classical Deep Reinforcement Learning (DRL) with neutral atom quantum processing using Pasqal's `Pulser` library and OpenAI `Gymnasium`.

## 1. Setup & Module Initialization
Run the following cell to initialize the environment. If you are on **Google Colab**, this will automatically clone the repository and install all required libraries.

In [ ]:
import os
import sys
import warnings
import numpy as np
import matplotlib.pyplot as plt

# --- SOTA Noise Suppression ---
warnings.filterwarnings("ignore")
try:
    from pyparsing import PyparsingDeprecationWarning
    warnings.filterwarnings("ignore", category=PyparsingDeprecationWarning)
except ImportError:
    pass

# --- Environment Setup (Colab / Local) ---
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/Pulser-gym'):
        print("Cloning repository...")
        !git clone -q https://github.com/charleshusson75-cell/Pulser-gym.git /content/Pulser-gym
    os.chdir('/content/Pulser-gym')
    print("Installing dependencies (this may take a few minutes)...")
    !pip install -q qutip pulser gymnasium stable-baselines3 shimmy tqdm rich

env_package_path = os.path.abspath('pulser-gym-env')
if env_package_path not in sys.path:
    sys.path.append(env_package_path)

from pulser import Register
from stable_baselines3 import PPO
from pulser_gym.env_01_core import PulserEnv
from pulser_gym.sequence_02_translation import build_pulse_sequence

print(f"✅ Active Directory: {os.getcwd()}")
print("✅ Framework and 2D Physical Bridge initialized successfully.")

## 2. Register Layout and Visualization
The 2D square lattice consists of 9 atoms arranged in a 3x3 grid with 5 $\mu$m spacing. The agent optimizes the state selection subject to the Rydberg blockade radius constraints.

In [ ]:
n_atoms = 9
side_length = int(np.sqrt(n_atoms))
graph_register = Register.square(side_length, spacing=5.0, prefix='q')

fig, ax = plt.subplots(figsize=(6, 4))
graph_register.draw(with_labels=True, custom_ax=ax)
ax.set_title(f"2D Lattice Topology ({side_length}x{side_length})")
plt.show()

## 3. Gymnasium Environment Configuration
The environment now defines a **2D continuous action space**:
1. **Amplitude**: [0, 1] scaled to [0, 10] rad/µs
2. **Detuning**: [0, 1] scaled to [-20, 20] rad/µs

In [ ]:
env = PulserEnv(n_qubits=n_atoms)

print(f"2D Action Space: {env.action_space}")
print(f"Observation Space: {env.observation_space}")

## 4. Training the PPO Agent
The agent optimizes both Amplitude and Detuning simultaneously over 1,024 timesteps.

In [ ]:
print("Commencing 2D optimization loop...")
model = PPO("MlpPolicy", env, verbose=0)
model.learn(total_timesteps=1024, progress_bar=True)

print("\nTraining complete.")

## 5. Evaluation and Waveform Generation
We extract the agent's deterministic choices for both control parameters.

In [ ]:
obs, _ = env.reset()
action, _ = model.predict(obs, deterministic=True)

generated_sequence = build_pulse_sequence(n_atoms, action)
amplitude = action[0] * 10.0
detuning = (action[1] * 40.0) - 20.0
final_obs, final_reward, _, _, _ = env.step(action)

print(f"--- Quantum Control Results ---")
print(f"Optimal Amplitude: {amplitude:.3f} rad/us")
print(f"Optimal Detuning:  {detuning:.3f} rad/us")
print(f"Output Bitstring:  {final_obs}")
print(f"Final Reward:      {final_reward:.1f}")

generated_sequence.draw()